# Amazon Search Ranking — Week 8 V1 Final Benchmark
**One-click Colab notebook**

Upload the 10 final Week 1–7 ZIP files when prompted. The notebook consolidates ranking, retrieval, cold-start, ablation, and deployment metrics without incorrectly mixing different evaluation scopes.

In [ ]:

# ============================================================
# AMAZON SEARCH RANKING — WEEK 8 V1 FINAL BENCHMARK (ONE CLICK)
# Upload the 10 Week 1–7 ZIP files when prompted.
# Output: amazon_week8_v1_final_benchmark_results.zip
# ============================================================

import os, zipfile, shutil, glob
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import files
    uploaded = files.upload()
    uploaded_paths = [os.path.abspath(name) for name in uploaded.keys()]
except Exception:
    uploaded_paths = glob.glob("*.zip")

required_patterns = [
    "week1_v4_processed_data",
    "week2_baseline_ranking_results",
    "week4_v1_sbert_faiss_results",
    "week4_v2_clip_multimodal_results",
    "week5_v2_lambdamart_results",
    "week5_v3_model_comparison_ablation_results",
    "week5_v4_listnet_results",
    "week6_v4_final_evaluation_results",
    "week7_v4_final_package",
    "week7_v1_end_to_end_results",
]

def pick(pattern):
    hits = [p for p in uploaded_paths if pattern.lower() in os.path.basename(p).lower()]
    if not hits:
        raise FileNotFoundError(f"Missing required ZIP: {pattern}")
    return hits[0]

Z = {pat: pick(pat) for pat in required_patterns}

def read_csv_zip(zp, member):
    with zipfile.ZipFile(zp) as z:
        with z.open(member) as f:
            return pd.read_csv(f)

def ap_at_k(df, rank_col, rel_col="relevance_label", k=10):
    vals = []
    for qid, g_all in df.groupby("query_id"):
        total_rel = int((g_all[rel_col].fillna(0).to_numpy() > 0).sum())
        if total_rel == 0:
            continue
        g = g_all.sort_values(rank_col).head(k)
        rel = (g[rel_col].fillna(0).to_numpy() > 0).astype(int)
        hits, s = 0, 0.0
        for i, r in enumerate(rel, 1):
            if r:
                hits += 1
                s += hits / i
        vals.append(s / min(total_rel, k))
    return float(np.mean(vals)) if vals else np.nan

OUT = Path("/content/week8_v1_final_benchmark_results")
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)

# ---- Main ranking benchmark ----
model_cmp = read_csv_zip(Z["week5_v3_model_comparison_ablation_results"],
                         "week5_v3_model_comparison.csv")
listnet_cmp = read_csv_zip(Z["week5_v4_listnet_results"],
                           "week5_v4_listnet_comparison.csv")
bm25_top = read_csv_zip(Z["week2_baseline_ranking_results"], "bm25_top10.csv")
tfidf_top = read_csv_zip(Z["week2_baseline_ranking_results"], "tfidf_top10.csv")
xgb_top = read_csv_zip(Z["week2_baseline_ranking_results"], "xgboost_top10.csv")
lm_rank = read_csv_zip(Z["week5_v2_lambdamart_results"], "week5_v2_test_rankings.csv")
ln_rank = read_csv_zip(Z["week5_v4_listnet_results"], "week5_v4_listnet_test_rankings.csv")

map10 = {
    "BM25": ap_at_k(bm25_top, "rank"),
    "TF-IDF": ap_at_k(tfidf_top, "rank"),
    "XGBoost Pointwise": ap_at_k(xgb_top, "rank"),
    "LambdaMART": ap_at_k(lm_rank, "predicted_rank"),
    "ListNet": ap_at_k(ln_rank, "predicted_rank"),
}
for score_col, model_name in [("sbert_similarity","SBERT"),("clip_similarity","CLIP")]:
    t = lm_rank.copy()
    t["_rank"] = t.groupby("query_id")[score_col].rank(method="first", ascending=False)
    map10[model_name] = ap_at_k(t, "_rank")

ranking = model_cmp[["model","NDCG@10","Recall@10","MRR"]].copy()
ranking = pd.concat([
    ranking,
    listnet_cmp.loc[listnet_cmp["model"].eq("ListNet"),
                    ["model","NDCG@10","Recall@10","MRR"]]
], ignore_index=True)
ranking["MAP@10"] = ranking["model"].map(map10)
ranking["Recall@50"] = np.nan
ranking["queries"] = 200
ranking["evaluation_scope"] = "Fixed 200-query reranking candidate set"
ranking["source"] = ranking["model"].map({
    "BM25":"Week2/Week5","TF-IDF":"Week2/Week5",
    "XGBoost Pointwise":"Week2/Week5","LambdaMART":"Week5 V2/V3",
    "SBERT":"Week5 V2/V3","CLIP":"Week5 V2/V3","ListNet":"Week5 V4"
})
ranking = ranking.sort_values("NDCG@10", ascending=False).reset_index(drop=True)
ranking.to_csv(OUT/"week8_v1_ranking_benchmark.csv", index=False)

# ---- Full-catalog retrieval ----
retr = read_csv_zip(Z["week4_v2_clip_multimodal_results"], "week4_v2_final_metrics.csv")
retrieval = retr.rename(columns={
    "NDCG@10_judged":"NDCG@10","MRR_judged":"MRR",
    "Recall@10_full":"Recall@10","Recall@50_full":"Recall@50",
    "queries_evaluated":"queries"
})[["model","NDCG@10","MRR","Recall@10","Recall@50","queries"]].copy()
retrieval["MAP@10"] = np.nan
retrieval["evaluation_scope"] = "Full-catalog retrieval; judged NDCG/MRR + full Recall"
retrieval["source"] = "Week4 V2"
retrieval.to_csv(OUT/"week8_v1_retrieval_benchmark.csv", index=False)

# ---- Cold-start holdout ----
cold = read_csv_zip(Z["week6_v4_final_evaluation_results"],
                    "week6_v4_final_holdout_metrics.csv")
coldstart = cold.rename(columns={
    "ndcg@10":"NDCG@10","ndcg@3":"NDCG@3","recall@10":"Recall@10",
    "recall@3":"Recall@3","mrr":"MRR","n_queries":"queries"
})[["model","queries","NDCG@10","NDCG@3","Recall@10","Recall@3","MRR",
    "ndcg10_improvement_vs_bm25_pct"]].copy()
coldstart["MAP@10"] = np.nan
coldstart["Recall@50"] = np.nan
coldstart["evaluation_scope"] = "45-query cold-start holdout"
coldstart["source"] = "Week6 V4"
coldstart = coldstart.sort_values("NDCG@10", ascending=False).reset_index(drop=True)
coldstart.to_csv(OUT/"week8_v1_coldstart_benchmark.csv", index=False)

# ---- Ablation + deployment ----
ablation = read_csv_zip(Z["week5_v3_model_comparison_ablation_results"],
                        "week5_v3_ablation_results.csv")
ablation.to_csv(OUT/"week8_v1_ablation_results.csv", index=False)

sys_metrics = read_csv_zip(Z["week7_v4_final_package"],
                           "final_evaluation/week7_v4_final_metrics.csv")
stage_latency = read_csv_zip(Z["week7_v4_final_package"],
                             "final_evaluation/week7_v4_stage_latency.csv")
sys_metrics.to_csv(OUT/"week8_v1_system_metrics.csv", index=False)
stage_latency.to_csv(OUT/"week8_v1_stage_latency.csv", index=False)

with zipfile.ZipFile(Z["week7_v4_final_package"]) as z:
    arch = "final_evaluation/week7_v4_architecture.png"
    if arch in z.namelist():
        with z.open(arch) as src, open(OUT/"week8_v1_system_architecture.png","wb") as dst:
            shutil.copyfileobj(src, dst)

# ---- Evaluation-scope notes ----
pd.DataFrame([
    ["ranking_benchmark","Fixed 200-query reranking candidate set",
     "Recall@10 saturates at 1.0; MAP@10 uses relevance_label > 0; Recall@50 intentionally N/A."],
    ["retrieval_benchmark","200-query full-catalog retrieval",
     "Recall@10/50 are full-catalog metrics; NDCG/MRR are judged; MAP@10 left N/A to avoid treating unjudged results as nonrelevant."],
    ["coldstart_benchmark","45-query cold-start holdout",
     "Separate split; do not present raw values as if evaluated on the same 200-query set."],
    ["system_metrics","Week 7 deployment/runtime validation",
     "Serving latency/architecture metrics, not ranking quality."]
], columns=["table","scope","important_note"]).to_csv(
    OUT/"week8_v1_metric_scope_notes.csv", index=False)

# ---- Plots ----
def save_barh(df, label, value, title, xlabel, name):
    plt.figure(figsize=(9,5))
    d = df.sort_values(value, ascending=True)
    plt.barh(d[label], d[value])
    plt.xlabel(xlabel)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(OUT/name, dpi=180)
    plt.close()

save_barh(ranking,"model","NDCG@10","Week 8 V1 — Main Ranking Benchmark",
          "NDCG@10","week8_v1_ranking_ndcg10.png")
save_barh(retrieval,"model","Recall@50","Week 8 V1 — Full-Catalog Retrieval Recall@50",
          "Recall@50","week8_v1_retrieval_recall50.png")
save_barh(coldstart,"model","NDCG@10","Week 8 V1 — Cold-Start Holdout Benchmark",
          "NDCG@10","week8_v1_coldstart_ndcg10.png")
save_barh(ablation,"ablation","NDCG@10","Week 8 V1 — LambdaMART Feature Ablation",
          "NDCG@10","week8_v1_ablation_ndcg10.png")
save_barh(stage_latency,"stage","mean_ms","Week 8 V1 — End-to-End Stage Latency",
          "Mean latency (ms)","week8_v1_stage_latency.png")

# ---- Summary ----
best = ranking.iloc[0]
bm25 = float(ranking.loc[ranking["model"].eq("BM25"),"NDCG@10"].iloc[0])
cold_lm = float(coldstart.loc[coldstart["model"].eq("LambdaMART"),"NDCG@10"].iloc[0])
cold_zs = float(coldstart.loc[coldstart["model"].eq("ZeroShot+LLM"),"NDCG@10"].iloc[0])
fresh = sys_metrics.loc[sys_metrics["metric"].eq("V1.3 fresh mean latency (ms)"),"value"]
fresh = float(fresh.iloc[0]) if len(fresh) else np.nan

summary = f"""WEEK 8 V1 — FINAL BENCHMARK SUMMARY

Main ranking:
- Best: {best['model']}
- NDCG@10: {best['NDCG@10']:.6f}
- MAP@10: {best['MAP@10']:.6f}
- MRR: {best['MRR']:.6f}
- Relative NDCG@10 gain vs BM25: {(best['NDCG@10']/bm25-1)*100:.2f}%

Cold-start:
- LambdaMART NDCG@10: {cold_lm:.6f}
- ZeroShot+LLM NDCG@10: {cold_zs:.6f}
- Gap: {cold_lm-cold_zs:.6f}

Deployment:
- V1.3 fresh mean latency: {fresh:.2f} ms

Important:
Main reranking, full-catalog retrieval, and cold-start use different evaluation scopes.
Keep them as separate benchmark panels in the final Week 8 report.
"""
(OUT/"week8_v1_executive_summary.txt").write_text(summary, encoding="utf-8")

pd.DataFrame(
    [{"file":p.name,"purpose":"Week 8 V1 benchmark/analysis output"} for p in sorted(OUT.iterdir())]
).to_csv(OUT/"week8_v1_manifest.csv", index=False)

zip_path = "/content/amazon_week8_v1_final_benchmark_results.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        z.write(p, arcname=p.name)

print("\n=== WEEK 8 V1 COMPLETE ===")
print(ranking[["model","NDCG@10","MAP@10","MRR","Recall@10"]].to_string(index=False))
print("\nOutput:", zip_path)

try:
    files.download(zip_path)
except Exception:
    pass
